In [24]:
from utils import load_manager

# man_path = 'rediska0123/ue_manager_gsm8k_Qwen2.5-Math-7B'
# scores_prm_path = 'configs/scores_prm_gsm8k_qwen7B.json'
# scores_reasoneval_path = 'configs/scores_reasoneval_gsm8k_qwen7B.json'
# has_final_ans = True

# man_path = 'rediska0123/ue_manager_gsm8k_Qwen2.5-Math-1.5B'
# scores_prm_path = 'configs/scores_prm_gsm8k_qwen1.5B.json'
# scores_reasoneval_path = 'configs/scores_reasoneval_gsm8k_qwen1.5B.json'
# has_final_ans = True

# man_path = 'rediska0123/ue_manager_proofnet_Qwen2.5-Math-7B'
# scores_prm_path = 'configs/scores_prm_proofnet_qwen7B.json'
# scores_reasoneval_path = 'configs/scores_reasoneval_proofnet_qwen7B.json'
# has_final_ans = False

man_path = 'rediska0123/ue_manager_proofnet_Qwen2.5-Math-1.5B'
scores_prm_path = 'configs/scores_prm_proofnet_qwen1.5B.json'
scores_reasoneval_path = 'configs/scores_reasoneval_proofnet_qwen1.5B.json'
has_final_ans = False

man = load_manager(man_path)
scores_prm = json.load(open(scores_prm_path, 'r'))
scores_reasoneval = json.load(open(scores_reasoneval_path, 'r'))

ue_manager.pth:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

In [25]:
## import numpy as np

def parse_ans(s, ignore_unfinished=False):
    if '####' in s:
        return float(s.split('####')[-1].replace(',', ''))
    if r'\boxed{' in s:
        x = s.split(r'\boxed{')[-1].split('}')[0]
        return float(''.join(a for a in x if a.isdigit()))
    return None

def print_test_stats(man, scores_prm):
    stats = man['stats']
    targets = np.array(man['gen_metrics']['claim', 'StepFactCheck'])
    
    acc, last_tgt = [], 0
    for t, h, prm in zip(stats['target_texts'], stats['greedy_texts'], scores_prm):
        if has_final_ans:
            at, ah = parse_ans(t), parse_ans(h)
            acc.append(np.isclose(at, ah) if ah is not None else 0)
        else:
            acc.append(all(t == 1 for t in targets[last_tgt:last_tgt + len(prm)]))
            last_tgt += len(prm)

    print('Skipping {} nan steps'.format(np.isnan(targets).sum()))
    print()
    targets = targets[~np.isnan(targets)].astype(int)
    print('Total problems:', len(stats['input_texts']))
    print('Correct answers: {} ({}%)'.format(sum(acc), round(100 * np.mean(acc), 2)))
    print('Incorrect answers: {} ({}%)'.format(len(acc) - sum(acc), round(100 - 100 * np.mean(acc), 2)))
    print()
    print('Total steps: {}'.format(len(targets)))
    print('Correct steps: {} ({}%)'.format(len(targets) - sum(targets), round(100 - 100 * np.mean(targets), 2)))
    print('Incorrect steps: {} ({}%)'.format(sum(targets), round(100 * np.mean(targets), 2)))

print_test_stats(man, scores_prm)

Skipping 0 nan steps

Total problems: 186
Correct answers: 4 (2.15%)
Incorrect answers: 182 (97.85%)

Total steps: 4861
Correct steps: 2978 (61.26%)
Incorrect steps: 1883 (38.74%)


In [26]:
import json
from utils import load_manager
from metrics import ROCAUC, PRAUC, ECE

def flatten(x):
    return [b for a in x for b in a]

estimations = man['estimations']
methods = {}
for (_, ue_name), ue_vals in estimations.items():
    methods[ue_name] = ue_vals

methods['Qwen2.5-Math-7B-PRM800K'] = [-y for y in flatten(scores_prm)]
methods['ReasonEval'] = [s['redundancy'] - s['validity'] for s in flatten(scores_reasoneval)]
methods['ReasonEval_validity'] = [-s['validity'] for s in flatten(scores_reasoneval)]
methods['ReasonEval_redundancy'] = [s['redundancy'] for s in flatten(scores_reasoneval)]

targets = man['gen_metrics']['claim', 'StepFactCheck']

for key, val in methods.items():
    print(f'{key}: {len(val)} values')

print(f'Targets: {len(targets)} values')

RandomBaselineClaim: 4861 values
MaximumClaimProbability: 4861 values
CCP_claim_fact_pref: 4861 values
MaxTokenEntropyClaim: 4861 values
PerplexityClaim: 4861 values
LuqClaimEstimatorDummy_claim: 4861 values
Qwen2.5-Math-7B-PRM800K: 4861 values
ReasonEval: 4861 values
ReasonEval_validity: 4861 values
ReasonEval_redundancy: 4861 values
Targets: 4861 values


In [27]:
import pandas as pd
from metrics import ROCAUC, PRAUC, ECE
from plot_utils import pretty_plot_table
from collections import defaultdict

metrics = [ROCAUC(), PRAUC(), ECE()]
res_df = defaultdict(dict)

def rename_method(s):
    if s == 'RandomBaselineClaim':
        return 'Random'
    if s == 'MaximumClaimProbability':
        return 'MaxProb'
    if s == 'MaxTokenEntropyClaim':
        return 'MaxEntropy'
    if s == 'PerplexityClaim':
        return 'Perplexity'
    if 'LuqClaimEstimator' in s:
        return 'UHead'
    return s

for method_nm, method_vals in methods.items():
    if len(method_vals) != len(targets):
        print(f'Skipping {method_nm}: inconsistent number of samples, '
              f'expected {len(targets)}, got {len(method_vals)}')
        continue
    for m in metrics:
        res_df[str(m)][rename_method(method_nm)] = m(method_vals, targets)
    
df = pd.DataFrame(res_df)
pretty_plot_table(df)

,roc-auc,pr-auc,ece
Random,0.493281,0.384855,0.263371
MaxProb,0.469187,0.382941,0.434053
CCP_claim_fact_pref,0.465169,0.381755,0.358474
MaxEntropy,0.456876,0.364085,0.380001
Perplexity,0.471853,0.371558,0.322102
UHead,0.581671,0.441409,0.289487
Qwen2.5-Math-7B-PRM800K,0.596961,0.466681,0.306056
ReasonEval,0.553895,0.466594,0.115531
ReasonEval_validity,0.589051,0.472624,0.145032
ReasonEval_redundancy,0.495423,0.399027,0.243141
